# Chapter 0 — Five dwarf galaxies in APOGEE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/chemical-evolution-tutorial/blob/main/notebooks/00_data.ipynb)

**Goal.** This tutorial models the chemical evolution of five dwarf galaxies: the Large and Small Magellanic Clouds
(LMC, SMC), the Sagittarius dwarf (Sgr), Fornax, and Gaia-Sausage/Enceladus (GSE), a dwarf that merged with the
Milky Way about 10 Gyr ago. Their red giants were observed by the APOGEE survey (data release 17). Hasselquist et
al. (2021, H21; https://doi.org/10.3847/1538-4357/ac25f9) selected the members of each galaxy and fitted one-zone
chemical-evolution models to them. This chapter loads those members, defines the quality cut used in every later
chapter, and draws the [α/Fe]–[Fe/H] tracks that the models must reproduce.

**You will learn**
1. where the samples come from, and how to check them against the paper;
2. what [Mg/Fe] and [Si/Fe] against [Fe/H] look like for five galaxies of different mass;
3. how to summarise a cloud of stars by a running median with an honest error band;
4. which parts of the data deserve doubt: membership, a sparse metal-poor tail, and [Fe/H] as a clock.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts attached to a GitHub release of the tutorial. MANIFEST.json lists, for each
# file, where it came from and its SHA-256 checksum. Inside a clone of the repository the files are read from
# ../data_build instead of being downloaded.
DATA_URL = 'https://github.com/vasilybelokurov/chemical-evolution-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def fetch(name, cache=Path('data')):
    """Return the local path of data file `name`, downloading it if needed; its SHA-256 must match the manifest."""
    want = MANIFEST['files'][name]['sha256']
    path = Path('../data_build') / name                        # inside a clone of the repository
    if not path.exists():
        path = cache / name                                    # elsewhere (e.g. Colab): download into ./data
        if not path.exists() or sha256(path) != want:
            cache.mkdir(exist_ok=True)
            print(f'downloading {name} ...')
            urllib.request.urlretrieve(DATA_URL + name, path)
    assert sha256(path) == want, f'{name}: checksum mismatch'
    return path


GALAXIES = {'LMC': 'lmc', 'SMC': 'smc', 'Sgr': 'sgr', 'GSE': 'gse', 'Fornax': 'fornax'}
COLORS = {'LMC': '#2a78d6', 'SMC': '#eb6834', 'Sgr': '#1baf7a', 'GSE': '#eda100', 'Fornax': '#e87ba4'}


def good(t, x):
    """The tutorial's quality cut (used in every chapter): for [Fe/H] and for [X/Fe] (x = 'mgfe' or 'sife'),
    the ASPCAP flag is clear, the value is finite and not a -9999 placeholder, and the error is positive."""
    ok = np.ones(len(t), bool)
    for c in ('feh', x):
        ok &= (t[f'{c}_flag'] == 0) & np.isfinite(t[c]) & (t[c] > -9000) & (t[f'{c}_err'] > 0)
    return ok


stars = {g: Table.read(fetch(f'{k}_h21.ecsv')) for g, k in GALAXIES.items()}
print({g: len(t) for g, t in stars.items()})

## 0.1 Where the stars come from

H21 published the APOGEE identifiers of their members in Table 2: 8016 rows, each an APOGEE ID and a galaxy.
We joined these IDs to the APOGEE DR17 catalogue (`allStarLite-dr17-synspec_rev1`) and kept one observation per
star; the manifest (`MANIFEST['files']`) records the origin of every file. You receive the result: one file per
galaxy, with [Fe/H], [Mg/Fe], [Si/Fe], their errors and flags, and a few stellar parameters.

A first check: do we have the same number of stars as the paper? H21 give the members per galaxy in their Table 1.

In [ ]:
table2 = Table.read(fetch('hasselquist2021_table2.ecsv'))
table2['Sys'] = [{'Fnx': 'Fornax'}.get(s, s) for s in table2['Sys']]   # Table 2 writes Fornax as 'Fnx'
h21_table1 = {'LMC': 3909, 'SMC': 1146, 'Sgr': 946, 'GSE': 972, 'Fornax': 192}   # H21, Table 1
for g, t in stars.items():
    n2 = int(np.sum(table2['Sys'] == g))
    print(f'{g:7s} Table 2: {n2:5d}   our file: {len(t):5d}   Table 1: {h21_table1[g]:5d}')
    assert len(t) == n2                                  # every Table-2 ID was found in DR17 (reproduction)
    if g != 'GSE':
        assert n2 == h21_table1[g]                       # Table 2 agrees with Table 1 (reproduction)
assert np.sum(table2['Sys'] == 'GSE') == 1823            # ... except for GSE: a documented failure

Every Table-2 ID was found in DR17, and for four galaxies Table 2 agrees with Table 1. **GSE does not:** Table 2
lists 1823 stars, Table 1 says 972. We do not know which 972 H21 used for their fits. We keep all 1823 and treat
GSE's membership as uncertain: GSE is spread over the whole sky and its members are chosen by orbit, so a
looser or tighter orbital cut changes the sample far more than for the four bound satellites.

A second check: is any star listed twice?

In [ ]:
ids, counts = np.unique(table2['ID'], return_counts=True)
twice = set(ids[counts > 1])
both = {g: np.isin(stars[g]['apogee_id'], list(twice)) for g in stars}
print({g: int(m.sum()) for g, m in both.items()})
assert len(twice) == 11 and both['GSE'].sum() == both['Sgr'].sum() == 11   # identity: 11 IDs in GSE and in Sgr

Eleven stars are members of both GSE and Sgr. Both galaxies are, or were, accreted onto the Milky Way halo, and
their debris overlaps in the sky and in velocity. We keep them in both samples, as H21 did, and check in §0.5 that
they do not change any result.

## 0.2 The quality cut

APOGEE's pipeline (ASPCAP) flags abundances it does not trust. Our cut, defined once in `good()` above and used in
every chapter, keeps a star for a given abundance ratio when the flags of [Fe/H] and of the ratio are clear, the
values are finite and the errors positive.

In [ ]:
for g, t in stars.items():
    print(f'{g:7s} {len(t):5d} stars;  pass for [Mg/Fe]: {good(t, "mgfe").sum():5d};  for [Si/Fe]: {good(t, "sife").sum():5d}')
n_mg = {g: good(t, 'mgfe').sum() for g, t in stars.items()}
assert n_mg == {'LMC': 3900, 'SMC': 1138, 'Sgr': 935, 'GSE': 1811, 'Fornax': 183}   # regression (development run)

The cut removes at most 12 stars per galaxy. Few stars fail because H21 already required good spectra.

## 0.3 The tracks

A **running median** summarises the cloud: in a window 0.1 dex wide in [Fe/H], moved in steps of 0.05 dex, take
the median [X/Fe] of the stars inside, if there are at least 20. The median is robust to the few stars with
bad abundances. Its uncertainty comes from the **bootstrap**: draw the window's stars again with replacement, 300
times, take the median each time, and quote the 16th–84th percentiles of those medians.

In [ ]:
def running_median(x, y, width=0.1, step=0.05, nmin=20, nboot=300, seed=1):
    """Median of y in sliding windows of x (width, step), where a window has >= nmin stars.
    Returns arrays: window centre, median, bootstrap 16% and 84% of the median, number of stars."""
    rng = np.random.default_rng(seed)
    rows = []
    for c in np.arange(x.min(), x.max() + step, step):
        s = y[np.abs(x - c) <= width / 2]                        # stars in this window
        if len(s) < nmin:
            continue
        boot = np.median(rng.choice(s, (nboot, len(s))), axis=1)  # nboot resamples, each the same size as s
        rows.append((c, np.median(s), *np.percentile(boot, [16, 84]), len(s)))
    return np.array(rows).T


def xy(g, x):
    """[Fe/H] and [X/Fe] of the stars of galaxy g that pass the quality cut for x."""
    t = stars[g][good(stars[g], x)]
    return np.asarray(t['feh'], float), np.asarray(t[x], float)


med = {(g, x): running_median(*xy(g, x)) for g in stars for x in ('mgfe', 'sife')}

fig, axes = plt.subplots(2, 3, figsize=(13, 7.5), sharex=True, sharey=True)
for ax, g in zip(axes.flat, stars):
    for other in stars:                                          # the other four galaxies in grey
        if other != g:
            ax.scatter(*xy(other, 'mgfe'), s=2, c='0.82', lw=0, rasterized=True)
    ax.scatter(*xy(g, 'mgfe'), s=4 if len(stars[g]) > 1000 else 9, c=COLORS[g], alpha=0.5, lw=0, rasterized=True)
    c, m, lo, hi, n = med[g, 'mgfe']
    ax.plot(c, m, color='k', lw=2)
    ax.text(0.96, 0.94, f'{g}\nN = {n_mg[g]}', transform=ax.transAxes, ha='right', va='top')
ax = axes.flat[5]
for g in stars:
    c, m, lo, hi, n = med[g, 'mgfe']
    ax.fill_between(c, lo, hi, color=COLORS[g], alpha=0.3, lw=0)
    ax.plot(c, m, color=COLORS[g], lw=2, label=g)
ax.legend(frameon=False, fontsize=9)
for ax in axes.flat:
    ax.set(xlim=(-2.3, 0.0), ylim=(-0.45, 0.55))
    ax.grid(color='0.9', lw=0.6)
for ax in axes[1]:
    ax.set_xlabel('[Fe/H]')
for ax in axes[:, 0]:
    ax.set_ylabel('[Mg/Fe]')
fig.tight_layout()

**Figure 0.1.** [Mg/Fe] against [Fe/H] (both in dex) for the members of each galaxy that pass the quality cut.
Panels 1–5: one galaxy in colour (LMC blue, SMC orange, Sgr green, GSE yellow, Fornax pink), the other four in grey,
and its running median (black; 0.1-dex windows, ≥ 20 stars). Panel 6: the five running medians with their bootstrap
16–84% bands.

Each galaxy draws its own track, and the tracks differ in level and in shape:
- **GSE** is the most α-rich: its median falls steadily from about +0.30 at [Fe/H] = −2 to 0.0 at −0.64.
- **Sgr** falls from +0.10 to −0.13, with a sharp drop near [Fe/H] = −0.9.
- **LMC** falls to 0.0 by [Fe/H] ≈ −1.4, stays flat, then **rises** again to +0.07 above [Fe/H] ≈ −0.8.
  **SMC** shows a smaller bump near −1.0 before falling.
- **Fornax** is the most α-poor, with [Mg/Fe] between −0.19 and −0.13 in its core, rising slightly with [Fe/H].

A decline is what a galaxy forming stars at a steady rate produces: iron from type Ia supernovae arrives late and
lowers [Mg/Fe]. A rise needs something else: a burst of star formation adds fresh core-collapse supernovae, rich
in Mg. Chapters 1–3 build both effects from a model with known input; Chapter 4 fits them.

In [ ]:
# regression: the track ends of the development run (2026-11-11): [Fe/H] range of the median, its max and min [Mg/Fe]
frozen = {'LMC': (-1.61, -0.41, 0.091, -0.005), 'SMC': (-1.57, -0.67, 0.030, -0.096),
          'Sgr': (-1.50, -0.20, 0.103, -0.126), 'GSE': (-1.99, -0.64, 0.298, 0.003),
          'Fornax': (-1.07, -0.77, -0.134, -0.185)}
for g, ref in frozen.items():
    c, m, *_ = med[g, 'mgfe']
    np.testing.assert_allclose([c.min(), c.max(), m.max(), m.min()], ref, atol=0.006)
# the LMC rise: the median above [Fe/H] = -0.9 climbs back by more than 0.05 dex over its flat part
c, m, lo, hi, n = med['LMC', 'mgfe']
rise = m[c > -0.9].max() - m[(c > -1.4) & (c < -0.8)].min()
print(f'LMC [Mg/Fe] rise above [Fe/H] = -0.9: {rise:.3f} dex (bootstrap half-width there ~{np.median((hi - lo)[c > -0.9]) / 2:.3f})')
assert rise > 0.05

## 0.4 Magnesium and silicon

Mg and Si are both α-elements, made mostly in core-collapse supernovae. H21 fitted their models to **[Si/Fe]**,
not [Mg/Fe], and so does Chapter 4. Figure 0.2 compares the two.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharex=True, sharey=True)
for ax, x, lab in zip(axes, ('mgfe', 'sife'), ('[Mg/Fe]', '[Si/Fe]')):
    for g in stars:
        c, m, lo, hi, n = med[g, x]
        ax.fill_between(c, lo, hi, color=COLORS[g], alpha=0.3, lw=0)
        ax.plot(c, m, color=COLORS[g], lw=2, label=g)
    ax.set(xlabel='[Fe/H]', ylabel=lab, xlim=(-2.3, 0.0), ylim=(-0.3, 0.4))
    ax.grid(color='0.9', lw=0.6)
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()

span = {x: {g: med[g, x][1].max() - med[g, x][1].min() for g in stars} for x in ('mgfe', 'sife')}
print('decline of the median (max - min), dex:')
for g in stars:
    print(f"  {g:7s} [Mg/Fe] {span['mgfe'][g]:.3f}   [Si/Fe] {span['sife'][g]:.3f}")
order = {x: sorted(stars, key=lambda g: np.median(med[g, x][1])) for x in ('mgfe', 'sife')}
print('galaxies ordered by median level:', order)
assert order['mgfe'][0] == order['sife'][0] == 'Fornax' and order['mgfe'][-1] == order['sife'][-1] == 'GSE'

**Figure 0.2.** Running medians (lines) and bootstrap 16–84% bands of [Mg/Fe] (left) and [Si/Fe] (right) against
[Fe/H], for the five galaxies (colours as in Figure 0.1).

The two elements agree on the essentials: GSE is the most α-rich and Fornax the least, at every [Fe/H] they share.
The details differ. The LMC bump above [Fe/H] ≈ −0.8 is weaker in [Si/Fe]. The total decline (printed above) is
larger in [Si/Fe] for the LMC and Sgr, about the same for the SMC, and smaller for GSE and Fornax. Type Ia
supernovae make some silicon but almost no magnesium, so a model predicts a shallower decline in [Si/Fe]; the data
agree with that only for GSE and Fornax. Keep this in mind: element-to-element differences of 0.05 dex can come from
the abundance analysis as well as from nucleosynthesis.

## 0.5 Doubts

**The Fornax tail.** Fornax has 183 good stars, but only 46 lie below [Fe/H] = −1.1: too few for 0.1-dex windows
of 20 stars. Instead we use windows of a fixed number of stars: sort the tail by [Fe/H], take the median of each
run of 10 neighbours, and slide by 2.

In [ ]:
def equal_count_median(x, y, k=10, step=2, nboot=300, seed=1):
    """Median of y in sliding windows of k stars ordered by x (step: stars). Returns median x, median y, 16%, 84%."""
    rng = np.random.default_rng(seed)
    o = np.argsort(x)
    x, y = x[o], y[o]
    rows = []
    for i in range(0, len(x) - k + 1, step):
        boot = np.median(rng.choice(y[i:i + k], (nboot, k)), axis=1)
        rows.append((np.median(x[i:i + k]), np.median(y[i:i + k]), *np.percentile(boot, [16, 84])))
    return np.array(rows).T


fx, fy = xy('Fornax', 'mgfe')
tail = fx < -1.1
tc, tm, tlo, thi = equal_count_median(fx[tail], fy[tail])
c, m, lo, hi, n = med['Fornax', 'mgfe']

fig, ax = plt.subplots(figsize=(7, 4.3))
ax.scatter(fx, fy, s=14, c=COLORS['Fornax'], alpha=0.6, lw=0)
ax.fill_between(c, lo, hi, color='k', alpha=0.15, lw=0)
ax.plot(c, m, color='k', lw=2, label='0.1-dex windows (≥ 20 stars)')
ax.fill_between(tc, tlo, thi, color=COLORS['Fornax'], alpha=0.3, lw=0)
ax.plot(tc, tm, color=COLORS['Fornax'], lw=2, ls='--', label='windows of 10 stars, step 2')
ax.axvline(-1.1, color='0.5', lw=0.8, ls=':')
ax.set(xlabel='[Fe/H]', ylabel='[Mg/Fe]', xlim=(-2.5, -0.3), ylim=(-0.6, 0.5))
ax.legend(frameon=False, fontsize=9)
ax.grid(color='0.9', lw=0.6)
print(f'{tail.sum()} stars below [Fe/H] = -1.1; {len(tc)} windows; lowest window at [Fe/H] = {tc.min():.2f}')
assert tail.sum() == 46 and len(tc) == 19 and abs(tc.min() + 1.96) < 0.006     # regression

**Figure 0.3.** Fornax: [Mg/Fe] against [Fe/H] for its 183 good stars (pink points). Black: the running median of
Figure 0.1 with its bootstrap band. Pink dashed: medians of 10-star windows of the 46 stars below [Fe/H] = −1.1
(dotted line), with bootstrap 16–84% bands.

The tail rises to [Mg/Fe] ≈ +0.07 below [Fe/H] = −1.8, but each point rests on 10 stars and its band is about 0.1
dex wide. Some of these stars may be Milky Way halo stars in front of Fornax: the halo is metal-poor too, and our
file has no velocities to separate them. Read the tail as a hint, not a measurement.

**Overlapping members.** Do the 11 stars that are in both GSE and Sgr matter? Recompute both medians without them.

In [ ]:
for g in ('GSE', 'Sgr'):
    t = stars[g][good(stars[g], 'sife') & ~both[g]]
    c2, m2, *_ = running_median(np.asarray(t['feh'], float), np.asarray(t['sife'], float))
    c, m, lo, hi, n = med[g, 'sife']
    shift = np.abs(np.interp(c, c2, m2) - m)             # change of the median, window by window
    half = (hi - lo) / 2                                 # the window's own bootstrap error
    i = np.argmax(shift / half)
    print(f'{g}: worst window [Fe/H] = {c[i]:.2f} ({n[i]:.0f} stars): shift {shift[i]:.3f} dex, '
          f'its error {half[i]:.3f} dex')
    assert np.all(shift <= half + 0.001)                 # sensitivity: within each window's own error (0.001 rounding)

Removing them moves each window's median by no more than that window's own bootstrap error. The largest effect is in
GSE's most metal-poor window, which has only 20 stars and contains two of the eleven. They change nothing we will
conclude.

**[Fe/H] is not a clock.** Figure 0.1 puts five galaxies on one [Fe/H] axis, and it is tempting to read it as
time. It is not: how fast [Fe/H] grows depends on how fast a galaxy turns gas into stars and how much metal it
loses in winds, and both differ between galaxies. Two tracks that cross at the same [Fe/H] did not cross at the
same time. Chapter 3 shows [Fe/H] against time for
models of all five galaxies.

## Summary

- The samples are H21's Table 2 members in APOGEE DR17. They reproduce H21's Table 1 counts for LMC, SMC, Sgr and
  Fornax; GSE has 1823 stars against 972 in Table 1.
- One quality cut, `good()`, is used throughout; it removes at most 12 stars per galaxy.
- GSE is the most α-rich and Fornax the least. GSE and Sgr decline; the LMC and SMC show bumps that a steady
  star-formation history cannot make. [Si/Fe], used for the fits, tells the same story as [Mg/Fe] with
  0.05-dex differences in detail.
- The Fornax metal-poor tail rests on 46 stars and may contain foreground; the 11 GSE/Sgr overlaps do not matter.

## Exercises

1. Change the window width of `running_median` to 0.2 dex and `nmin` to 50. Which features of Figure 0.1 survive?
2. Plot [Si/Fe] − [Mg/Fe] against [Fe/H] for the five galaxies. Where does it rise, and why might it?
3. Plot the [Fe/H] distribution of each galaxy (a histogram). Which galaxy reaches the highest [Fe/H], and does that
   agree with its stellar mass?